# 01 — Getting Started with AG2 Beta

Build an agent, give it search tools, and use an **observer** to capture every citation the search returns — then render real hyperlinks.

Set `OPENAI_API_KEY` and `EXA_API_KEY` in your `.env`.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

In [ ]:
from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import ToolResultsEvent
from autogen.beta.tools import ExaToolkit

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

exa = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))

## Observer pattern — capture URLs live

`@agent.observer(ToolResultsEvent)` registers a callback (sync or async) that fires every time the agent receives tool results. We use it to record every `title → url` pair the search returns — before the agent even responds.

In [ ]:
title_to_url: dict[str, str] = {}

agent = Agent(
    "surveyor",
    prompt="You are a research surveyor. Search broadly, cite real sources.",
    config=config,
    tools=[exa],
)


@agent.observer(ToolResultsEvent)
def capture_urls(event: ToolResultsEvent) -> None:
    """Capture title→url from every search result, live."""
    for r in event.results:
        result = getattr(r, "result", None)
        if result is None:
            continue
        for part in getattr(result, "parts", []):
            data = getattr(part, "data", None)
            if data is None:
                continue
            for hit in getattr(data, "results", None) or []:
                title = getattr(hit, "title", None)
                url = getattr(hit, "url", None)
                if title and url:
                    title_to_url[title] = url

## Ask the agent

We use a shared `MemoryStream` so we can walk the events afterward.

In [ ]:
from IPython.display import Markdown, display

stream = MemoryStream()

reply = await agent.ask(
    "Find 5 recent papers on reactive event-driven multi-agent coordination. "
    "For each, give title, year, and one key finding.",
    stream=stream,
)

display(Markdown(reply.body))

## Captured URLs

The observer fired during the turn. Every search hit is now in `title_to_url`.

In [ ]:
lines = [f"**{len(title_to_url)} URLs captured by observer:**", ""]
for title, url in title_to_url.items():
    lines.append(f"- [{title}]({url})")
display(Markdown("\n".join(lines)))

## Walking the stream

The stream records every event. Below we render each type with rich formatting — search results show titles with real hyperlinks, answer results show citations.

In [ ]:
import json

from autogen.beta.events import (
    ModelRequest,
    ModelResponse,
    ToolCallEvent,
    ToolResultsEvent,
)


def _date(d):
    return f" \u00b7 {d[:10]}" if d else ""


def _trim(s, n):
    if not s:
        return ""
    s = s.replace("\n", " ").strip()
    return s if len(s) <= n else s[:n] + "\u2026"


def render_tool_data(data):
    """Render Exa results using duck typing (no internal imports needed)."""
    # Search response — has .results list of hits
    results = getattr(data, "results", None)
    if isinstance(results, list) and results and hasattr(results[0], "url"):
        lines = [f"_{len(results)} results_"]
        for i, r in enumerate(results, 1):
            title = getattr(r, "title", None) or getattr(r, "url", "")
            url = getattr(r, "url", "")
            date = _date(getattr(r, "published_date", None))
            lines.append(f"  {i}. [{title}]({url}){date}")
            snippet = _trim(getattr(r, "text", None), 180)
            if snippet:
                lines.append(f"     > {snippet}")
        return "\n".join(lines)

    # Answer result — has .answer and .citations
    answer = getattr(data, "answer", None)
    citations = getattr(data, "citations", None)
    if answer and citations:
        body = [answer, "", f"_{len(citations)} citations_"]
        for i, c in enumerate(citations, 1):
            body.append(f"  {i}. [{getattr(c, 'title', '') or c.url}]({c.url})")
            snippet = _trim(getattr(c, "text", None), 120)
            if snippet:
                body.append(f"     > {snippet}")
        return "\n".join(body)

    return f"`{data!r}`"


tool_calls: dict = {}

for ev in await stream.history.get_events():
    if isinstance(ev, ModelRequest):
        display(Markdown(f"**User** &nbsp; {ev.parts[0].content}"))

    elif isinstance(ev, ToolCallEvent):
        args = json.loads(ev.arguments) if isinstance(ev.arguments, str) else ev.arguments
        tool_calls[ev.id] = {"name": ev.name, "args": args}

    elif isinstance(ev, ToolResultsEvent):
        for r in ev.results:
            call = tool_calls.get(r.parent_id, {})
            name = call.get("name", "?")
            args = call.get("args", {})
            arg_str = ", ".join(f"`{k}`={v!r}" for k, v in args.items())
            data = r.result.parts[0].data
            display(
                Markdown(f"&nbsp;&nbsp;**`{name}`** &nbsp; {arg_str}\n\n{render_tool_data(data)}")
            )

    elif isinstance(ev, ModelResponse):
        if ev.content:
            display(Markdown(f"**Model** &nbsp; {ev.content}"))

## Up next

lionag2's agents don't just return text — they emit **typed events** (`FindingEmitted`, `DepthRequested`, etc.) that drive reactive coordination. Tutorial 02 introduces the event system and structured output.